# 01 — Data Audit

**Phase 2.** Forensic inspection of the authoritative EduPro workbook.

This notebook is an **explorable view** of the audit, not its implementation. Every
computation here comes from the `edupro` package or from `scripts/run_data_audit.py`,
so nothing shown can drift from what the pipeline and the research paper use
(CLAUDE.md §19: notebooks must never be the only implementation).

The raw workbook is opened read-only and its checksum verified (ADR-0003).

In [1]:
import json

import pandas as pd

from edupro import config
from edupro.data.joins import build_interactions, course_popularity
from edupro.data.loader import load_all, verify_raw_workbook
from edupro.data.validation import validate

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

checksum = verify_raw_workbook()
print(f"workbook verified: {checksum[:16]}…")

data = load_all()
interactions = build_interactions(data, with_user_demographics=True, with_teacher=True)
data

workbook verified: ed555e4613e6a210…


EduProData(users=3000, teachers=60, courses=60, transactions=10000)

## Step 1 — Profile every sheet

Note that `UserName`, `Email` and `TeacherName` are **absent**: PII is dropped at
load time rather than filtered downstream, so it is never present in a frame that
could reach a feature matrix or a figure (ADR-0006).

In [2]:
for name, frame in (
    (config.SHEET_USERS, data.users),
    (config.SHEET_TEACHERS, data.teachers),
    (config.SHEET_COURSES, data.courses),
    (config.SHEET_TRANSACTIONS, data.transactions),
):
    print(f"\n{name}: {frame.shape[0]:,} rows x {frame.shape[1]} cols  "
          f"(full-row duplicates: {frame.duplicated().sum()})")
    profile = pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "nulls": frame.isna().sum(),
        "unique": frame.nunique(),
    })
    print(profile.to_string())


Users: 3,000 rows x 3 cols  (full-row duplicates: 0)
         dtype  nulls  unique
UserID  string      0    3000
Age      int64      0      21
Gender  string      0       2

Teachers: 60 rows x 6 cols  (full-row duplicates: 0)
                     dtype  nulls  unique
TeacherID           string      0      60
Age                  int64      0      21
Gender              string      0       2
Expertise           string      0      12
YearsOfExperience    int64      0      16
TeacherRating      float64      0      56

Courses: 60 rows x 8 cols  (full-row duplicates: 0)
                  dtype  nulls  unique
CourseID         string      0      60
CourseName       string      0      58
CourseCategory   string      0      12
CourseType       string      0       2
CourseLevel      string      0       3
CoursePrice     float64      0      23
CourseDuration  float64      0      60
CourseRating    float64      0      56

Transactions: 10,000 rows x 7 cols  (full-row duplicates: 0)
            

## Step 2 — Data integrity (EXP-001)

Validation returns every finding at once rather than raising on the first, and it
never repairs anything: cleaning decisions must be explicit and recorded (§8).

In [3]:
report = validate(data)
print(f"{report.summary()}  ->  {'PASS' if report.ok else 'FAIL'}\n")
for finding in report.findings:
    print(finding)

0 error(s), 1 warning(s), 4 info  ->  PASS

[INFO   ] Transactions.repeat_enrollment: no repeat (UserID, CourseID) pairs: the signal is purely binary/implicit
[INFO   ] Transactions.date_validity: dates span 2025-01-01 to 2025-12-30 (358 distinct days)
[INFO   ] Transactions.amount_vs_price: Amount equals CoursePrice for every transaction: spending carries no information beyond which courses were taken
[INFO   ] Courses.type_vs_price: CourseType is a deterministic function of CoursePrice (Free <=> price == 0), so the two are redundant
[WARNING] Courses.duplicate_course_names: 2 repeated course name(s): ['Deep Learning', 'Natural Language Processing'] — distinct courses sharing a title, so titles cannot identify a course


In [4]:
# Referential integrity, stated explicitly.
checks = {
    "tx.UserID not in Users": (~data.transactions.UserID.isin(data.users.UserID)).sum(),
    "tx.CourseID not in Courses": (~data.transactions.CourseID.isin(data.courses.CourseID)).sum(),
    "tx.TeacherID not in Teachers": (~data.transactions.TeacherID.isin(data.teachers.TeacherID)).sum(),
    "Users never transacting": (~data.users.UserID.isin(data.transactions.UserID)).sum(),
    "Courses never enrolled": (~data.courses.CourseID.isin(data.transactions.CourseID)).sum(),
    "Teachers never appearing": (~data.teachers.TeacherID.isin(data.transactions.TeacherID)).sum(),
    "duplicate (User, Course) pairs": data.transactions.duplicated(["UserID", "CourseID"]).sum(),
}
pd.Series(checks, name="count").to_frame()

,count
tx.UserID not in Users,0
tx.CourseID not in Courses,0
tx.TeacherID not in Teachers,0
Users never transacting,0
Courses never enrolled,0
Teachers never appearing,0
"duplicate (User, Course) pairs",0


### EXP-002 — is `Amount` just `CoursePrice`?

Open question Q-1 from Phase 0. If they are identical then "average spending" and
"average course price" are one feature wearing two names.

In [5]:
merged = data.transactions.merge(
    data.courses[["CourseID", "CoursePrice", "CourseType"]], on="CourseID", how="left"
)
matches = (merged.Amount.round(2) == merged.CoursePrice.round(2))
print(f"Amount == CoursePrice on {matches.sum():,} / {len(merged):,} rows "
      f"({matches.mean() * 100:.2f}%)")
print(f"\ndistinct Amounts: {data.transactions.Amount.nunique()}  "
      f"distinct CoursePrices: {data.courses.CoursePrice.nunique()}")
print("\nprice by course type:")
print(data.courses.groupby("CourseType").CoursePrice.agg(["count", "min", "max", "mean"]))

Amount == CoursePrice on 10,000 / 10,000 rows (100.00%)

distinct Amounts: 23  distinct CoursePrices: 23

price by course type:
            count   min    max        mean
CourseType                                
Free           38  0.00    0.0    0.000000
Paid           22  0.78  490.9  253.599091


### EXP-005 — is `TeacherID` an alias for `CourseID`?

Open question Q-9. Phase 1 hypothesised a bijection (60 teachers, 60 courses), which
would have made the entire teacher experiment vacuous. This check gates EXP-014.

In [6]:
teachers_per_course = data.transactions.groupby("CourseID").TeacherID.nunique()
courses_per_teacher = data.transactions.groupby("TeacherID").CourseID.nunique()
pairs = data.transactions[["CourseID", "TeacherID"]].drop_duplicates()

print(f"distinct (course, teacher) pairs : {len(pairs):,}")
print(f"teachers per course : min={teachers_per_course.min()} max={teachers_per_course.max()}")
print(f"courses per teacher : min={courses_per_teacher.min()} max={courses_per_teacher.max()}")
print(f"\nbijection? {teachers_per_course.max() == 1 and courses_per_teacher.max() == 1}")
print("-> Phase 1 hypothesis REFUTED; EXP-014 proceeds.")

distinct (course, teacher) pairs : 887
teachers per course : min=7 max=30
courses per teacher : min=7 max=55

bijection? False
-> Phase 1 hypothesis REFUTED; EXP-014 proceeds.


## Step 6 — Sparsity (EXP-003)

The finding that shapes the entire recommendation design.

In [7]:
counts = data.transactions.groupby("UserID").size()
distribution = counts.value_counts().sort_index()
table = pd.DataFrame({
    "learners": distribution,
    "pct": (distribution / len(counts) * 100).round(2),
    "cumulative_pct": (distribution.cumsum() / len(counts) * 100).round(2),
})
table.index.name = "interactions"
print(table.to_string())
print(f"\nmean={counts.mean():.4f}  median={counts.median():.0f}  max={counts.max()}")
print(f"exactly 1 : {(counts == 1).sum():,} ({(counts == 1).mean() * 100:.1f}%)")
print(f"exactly 2 : {(counts == 2).sum():,}")
print(f"3 or more : {(counts >= 3).sum():,}")
missing = [n for n in range(1, counts.max() + 1) if n not in distribution.index]
print(f"\ninteraction counts with ZERO learners: {missing}")

              learners    pct  cumulative_pct
interactions                                 
1                 1620  54.00           54.00
2                  612  20.40           74.40
3                  186   6.20           80.60
4                  128   4.27           84.87
9                    1   0.03           84.90
10                   4   0.13           85.03
11                  31   1.03           86.07
12                  72   2.40           88.47
13                 122   4.07           92.53
14                 132   4.40           96.93
15                  74   2.47           99.40
16                  18   0.60          100.00

mean=3.3333  median=1  max=16
exactly 1 : 1,620 (54.0%)
exactly 2 : 612
3 or more : 768

interaction counts with ZERO learners: [5, 6, 7, 8]


## EXP-004 — temporal split viability

The pre-registered rule (`research/recommendation_evaluation_plan.md` §2.3): if the
global temporal split leaves fewer than **300** evaluable learners, the primary
protocol switches to leave-one-out and every figure is labelled leakage-bearing.

In [8]:
from edupro.evaluation.splits import (
    PROTOCOL_A_MIN_EVALUABLE, apply_global_split, global_temporal_split, leave_one_out_split,
)

split = global_temporal_split(interactions)
frames = apply_global_split(interactions, split)
loo = leave_one_out_split(interactions)

print(f"validation cut : {split.validation_date.date()}")
print(f"test cut       : {split.test_date.date()}")
print(f"partition sizes: " + ", ".join(f"{k}={len(v):,}" for k, v in frames.items()))
print(f"\nProtocol A evaluable learners: {split.n_evaluable_learners:,} "
      f"(threshold {PROTOCOL_A_MIN_EVALUABLE})")
print(f"Protocol A viable: {split.protocol_a_viable}")
print(f"\nProtocol B (leave-one-out): {len(loo['test']):,} evaluable, "
      f"{len(loo['excluded_single_interaction']):,} single-interaction learners excluded")

validation cut : 2025-09-12
test cut       : 2025-10-18
partition sizes: train=6,992, validation=1,000, fit=7,992, test=2,008

Protocol A evaluable learners: 791 (threshold 300)
Protocol A viable: True

Protocol B (leave-one-out): 1,380 evaluable, 1,620 single-interaction learners excluded


## Full machine-readable audit

`scripts/run_data_audit.py` writes every number above — plus the permutation-based
signal detection — to `artifacts/phase2_audit.json`. The narrative in
`research/dataset_audit.md` is written from that file.

In [9]:
audit = json.loads((config.ARTIFACTS_DIR / "phase2_audit.json").read_text(encoding="utf-8"))
print("sections:")
for key in audit:
    print(f"  {key}")
print("\nsignal detection — course choice vs popularity-matched null:")
for name, result in audit["signal_detection"]["preference_vs_popularity_null"].items():
    verdict = "SIGNAL" if result["signal"] else "no signal"
    print(f"  {name:<28} z={result['z']:+6.2f}  {verdict}")

sections:
  provenance
  EXP-001_validation
  sheet_profiles
  EXP-002_amount_vs_price
  EXP-003_sparsity
  EXP-004_temporal_split
  EXP-005_teacher_course_mapping
  EXP-006_distributions
  signal_detection
  feature_candidates

signal detection — course choice vs popularity-matched null:
  mean_distinct_categories     z= -1.17  no signal
  mean_top_category_share      z= +1.12  no signal
  mean_distinct_levels         z= -2.65  SIGNAL
  mean_free_share              z= +1.51  no signal
